In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "serverless_stable_ysmqdz_catalog"  # e.g. "main"
schema  = "abac_test"   # e.g. "abac_demo"

print(f"✓ Using: {catalog}.{schema}")


# Tutorial 1: Row Filtering with ABAC

This tutorial demonstrates how to use ABAC to filter table rows based on governed tags and group membership.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the catalog/schema to create policies
- `APPLY TAG` and `ASSIGN` privilege on governed tags
- Account groups: `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_admins`

## Setup

> **Note:** This tutorial uses `{catalog}` as the catalog name. Replace with your own catalog if needed.

### Create account groups

This tutorial references: `abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_admins`

Account-level groups must be created in the **Account Console** (Admin > Groups), not via SQL.

> If these groups already exist, skip this step.

### Create governed tags

Create the following key-only governed tag in the Catalog Explorer UI (**Catalog** > **Governed Tags** > **Create governed tag**):

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_region` | *(none — key-only tag)* |

> If this tag already exists, skip this step.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.customers (
  id INT, name STRING, email STRING, region STRING, revenue DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.customers VALUES
  (1,  'Alice Johnson',    'alice@acme.com',       'us',   50000.00),
  (2,  'Bob Smith',        'bob@acme.com',         'us',   120000.00),
  (3,  'Carol White',      'carol@acme.com',       'us',   500000.00),
  (4,  'Hans Mueller',     'hans@firma.de',        'eu',   45000.00),
  (5,  'Sophie Dubois',    'sophie@societe.fr',    'eu',   95000.00),
  (6,  'Lars Eriksson',    'lars@foretag.se',      'eu',   320000.00),
  (7,  'Yuki Tanaka',      'yuki@kaisha.jp',       'apac', 38000.00),
  (8,  'Wei Chen',         'wei@gongsi.cn',        'apac', 110000.00),
  (9,  'Priya Sharma',     'priya@company.in',     'apac', 275000.00),
  (10, 'James Wilson',     'james@corp.com',       'us',   88000.00),
  (11, 'Maria Garcia',     'maria@empresa.es',     'eu',   62000.00),
  (12, 'Kenji Watanabe',   'kenji@kigyou.jp',      'apac', 73000.00)
""")


In [ ]:
spark.sql(f"""
-- Grant base access so group members can query the tables.
-- ABAC policies handle the fine-grained row/column filtering on top of these grants.
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"""
-- Tag the region COLUMN so MATCH COLUMNS can find it
ALTER TABLE {catalog}.{schema}.customers ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")


## Step 1: Create the Row Filter UDF

This UDF receives a row's region value and returns TRUE if the current user belongs to the matching regional group.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.region_filter(region_val STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN region_val = 'us'   AND is_account_group_member('abac_tut_us_team')   THEN TRUE
  WHEN region_val = 'eu'   AND is_account_group_member('abac_tut_eu_team')   THEN TRUE
  WHEN region_val = 'apac' AND is_account_group_member('abac_tut_apac_team') THEN TRUE
  ELSE FALSE
END
""")


## Step 2: Create the Row Filter Policy

The policy uses `MATCH COLUMNS` to find columns tagged `abac_region` and passes their value to the UDF via `USING COLUMNS`.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE POLICY region_row_filter
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.region_filter
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")


## Step 3: EXCEPT Clause

The `EXCEPT` clause exempts specific groups from the policy. Here we exempt `abac_tut_admins` so they see all rows.

In [ ]:
spark.sql(f"""
-- Admins now see all 12 rows regardless of region
SELECT * FROM {catalog}.{schema}.customers
""")


## Step 4: Hierarchical Scope

Policies can be set at catalog, schema, or table level. A catalog-level policy applies to all tables in all schemas. Here we promote the region filter to catalog scope and show it applying across schemas.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.orders (
  order_id INT, customer_name STRING, region STRING, amount DOUBLE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.orders VALUES
  (101, 'Alice Johnson', 'us', 2500.00), (102, 'Hans Mueller', 'eu', 1800.00),
  (103, 'Yuki Tanaka', 'apac', 3200.00), (104, 'Bob Smith', 'us', 4100.00),
  (105, 'Sophie Dubois', 'eu', 2900.00)
""")

spark.sql(f"""
-- Tag the region column on the orders table too
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")


In [ ]:
# No additional grants needed — orders is in the same schema as customers


In [ ]:
spark.sql(f"""
-- Create catalog-level policy
CREATE OR REPLACE POLICY region_row_filter_catalog
ON CATALOG {catalog}
ROW FILTER {catalog}.{schema}.region_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")


## Expected Results

| User Group | Customers Visible | Orders Visible |
|------------|-------------------|----------------|
| `abac_tut_us_team` | US rows (1, 2, 3, 10) | US orders (101, 104) |
| `abac_tut_eu_team` | EU rows (4, 5, 6, 11) | EU orders (102, 105) |
| `abac_tut_apac_team` | APAC rows (7, 8, 9, 12) | APAC orders (103) |
| `abac_tut_admins` | All 12 rows | All 5 orders |

### Delete governed tags

If you no longer need it, delete `abac_region` from the Catalog Explorer UI (**Governed Tags** section).

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced in this section (`abac_tut_us_team`, `abac_tut_eu_team`, `abac_tut_apac_team`, `abac_tut_us_team`, `abac_tut_admins`, `abac_tut_admins`, `abac_tut_finance_team`, `abac_tut_hr_team`) must be created in the **Account Console** by an admin.
>
> Governed tags `abac_tut_region` and `abac_tut_emergency` (key-only) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Healthcare — Time-Based and Role-Based Row Filtering

Hospitals must restrict access to patient visit data based on staff region and role. On-call staff need emergency records even outside business hours, while standard staff only access their regional patients during business hours.

**Compliance context:** HIPAA requires minimum necessary access — staff should only see patient records relevant to their care responsibilities.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.hospital_visits (
  visit_id INT,
  patient_id INT,
  patient_name STRING,
  region STRING,
  department STRING,
  visit_date DATE,
  is_emergency BOOLEAN,
  attending_physician STRING,
  diagnosis_code STRING
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.hospital_visits VALUES
  (1, 101, 'Alice Brown',    'north', 'cardiology', '2026-01-15', FALSE, 'Dr. Smith', 'I21.0'),
  (2, 102, 'Bob Davis',      'south', 'neurology',  '2026-01-16', TRUE,  'Dr. Jones', 'G35'),
  (3, 103, 'Carol Evans',    'north', 'oncology',   '2026-01-17', FALSE, 'Dr. Lee',   'C34.1'),
  (4, 104, 'David Garcia',   'east',  'cardiology', '2026-01-18', TRUE,  'Dr. Patel', 'I50.9'),
  (5, 105, 'Eva Harris',     'west',  'neurology',  '2026-01-19', FALSE, 'Dr. Kim',   'G43.9'),
  (6, 106, 'Frank Iyer',     'south', 'oncology',   '2026-01-20', FALSE, 'Dr. Chen',  'C50.9'),
  (7, 107, 'Grace Johnson',  'east',  'cardiology', '2026-01-21', TRUE,  'Dr. Wong',  'I10'),
  (8, 108, 'Henry Kumar',    'west',  'neurology',  '2026-01-22', FALSE, 'Dr. Gupta', 'G20')
""")


In [ ]:
spark.sql(f"""
-- Tag region column for ABAC policy matching
ALTER TABLE {catalog}.{schema}.hospital_visits ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")

spark.sql(f"""
-- Tag is_emergency column for emergency access policy
ALTER TABLE {catalog}.{schema}.hospital_visits ALTER COLUMN is_emergency SET TAGS ('abac_tut_emergency' = '')
""")


### Healthcare Filter 1: Business Hours Access

Standard staff can only access patient records during business hours (7AM–7PM, Monday–Friday). This is enforced server-side by the UDF using `CURRENT_TIMESTAMP()`.

In [ ]:
spark.sql(f"""
-- Business hours filter: 7AM-7PM Monday-Friday
-- DAYOFWEEK: 1=Sunday, 2=Monday, ..., 6=Friday, 7=Saturday
CREATE OR REPLACE FUNCTION {catalog}.{schema}.business_hours_filter(dummy STRING)
RETURNS BOOLEAN
RETURN (HOUR(CURRENT_TIMESTAMP()) BETWEEN 7 AND 19)
  AND (DAYOFWEEK(CURRENT_DATE()) BETWEEN 2 AND 6)
""")


### Healthcare Filter 2: Emergency Access Override

On-call staff (members of `abac_tut_admins`) can always access emergency records regardless of business hours. Non-senior staff only see emergency visits.

In [ ]:
spark.sql(f"""
-- Emergency access: show only emergency=TRUE rows unless user is senior staff
CREATE OR REPLACE FUNCTION {catalog}.{schema}.emergency_access_filter(is_emergency BOOLEAN)
RETURNS BOOLEAN
RETURN is_emergency = TRUE OR is_account_group_member('abac_tut_admins')
""")


### Healthcare Filter 3: Regional Access Control

Each regional team (north, south, east, west) can only see patients from their own region. Admins are exempted via the EXCEPT clause.

In [ ]:
spark.sql(f"""
-- Regional access: staff see only patients in their region
CREATE OR REPLACE FUNCTION {catalog}.{schema}.abac_hc_region_filter(region_val STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN region_val = 'north' AND is_account_group_member('abac_tut_us_team') THEN TRUE
  WHEN region_val = 'south' AND is_account_group_member('abac_tut_eu_team') THEN TRUE
  WHEN region_val = 'east'  AND is_account_group_member('abac_tut_apac_team')  THEN TRUE
  WHEN region_val = 'west'  AND is_account_group_member('abac_tut_us_team')  THEN TRUE
  ELSE FALSE
END
""")


### Healthcare Policy: Schema-Level Regional Row Filter

Apply the regional filter at schema level. The `EXCEPT abac_tut_admins` clause ensures hospital administrators see all patient records across all regions.

In [ ]:
spark.sql(f"""
-- Schema-level regional row filter policy
-- Uses MATCH COLUMNS to automatically find columns tagged abac_tut_region
CREATE OR REPLACE POLICY abac_hc_regional_access
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.abac_hc_region_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS region_col
USING COLUMNS (region_col)
""")


In [ ]:
spark.sql(f"""
-- Verify: as a north team member, you should only see visits 1 and 3
-- As abac_tut_admins, you see all 8 visits
SELECT visit_id, patient_name, region, department, is_emergency
FROM {catalog}.{schema}.hospital_visits
ORDER BY region, visit_id
""")


### Healthcare Policy: Emergency Access Filter

Apply the emergency access filter using the `abac_tut_emergency` tag on the `is_emergency` column.

In [ ]:
spark.sql(f"""
-- Emergency access row filter policy
-- Non-senior staff can only see emergency visits (is_emergency = TRUE)
CREATE OR REPLACE POLICY abac_hc_emergency_access
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.emergency_access_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_emergency') AS emerg_col
USING COLUMNS (emerg_col)
""")


**Expected results:**

| Group | Visible Visits |
|-------|----------------|
| `abac_tut_us_team` | Visits 1, 3 (north region, both policies apply — regional first) |
| `abac_tut_admins` | Emergency visits 2, 4, 7 plus any regional access |
| `abac_tut_admins` | All 8 visits (exempt from both policies) |
| Non-member | No visits (fail-closed) |

---
## Financial Services — Fraud Detection Row Filtering

Fraud analysts need to see all flagged transactions across regions. Compliance officers need visibility into high-value transactions regardless of fraud status. Regional operations teams only see their own region's transactions.

**Compliance context:** SOX and PCI-DSS require audit trails and restricted access to transaction data. Only authorized roles should have access to transaction records.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.transactions (
  txn_id      INT,
  account_id  STRING,
  customer_name STRING,
  region      STRING,
  amount      DOUBLE,
  currency    STRING,
  txn_type    STRING,
  fraud_flag  BOOLEAN,
  risk_score  INT,
  txn_date    DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.transactions VALUES
  (1001, 'ACC-001', 'Alice Brown',   'us_east', 450.00,    'USD', 'purchase', FALSE, 12,  '2026-01-10'),
  (1002, 'ACC-002', 'Bob Davis',     'us_west', 12500.00,  'USD', 'wire',     FALSE, 35,  '2026-01-11'),
  (1003, 'ACC-003', 'Carol Evans',   'eu',      880.50,    'EUR', 'purchase', TRUE,  91,  '2026-01-12'),
  (1004, 'ACC-004', 'David Garcia',  'us_east', 25000.00,  'USD', 'wire',     TRUE,  88,  '2026-01-13'),
  (1005, 'ACC-005', 'Eva Harris',    'apac',    320.00,    'SGD', 'purchase', FALSE, 8,   '2026-01-14'),
  (1006, 'ACC-006', 'Frank Iyer',    'eu',      75000.00,  'EUR', 'transfer', FALSE, 42,  '2026-01-15'),
  (1007, 'ACC-007', 'Grace Johnson', 'us_west', 1200.00,   'USD', 'purchase', TRUE,  79,  '2026-01-16'),
  (1008, 'ACC-008', 'Henry Kumar',   'apac',    18000.00,  'USD', 'wire',     FALSE, 55,  '2026-01-17'),
  (1009, 'ACC-009', 'Iris Lee',      'us_east', 5.99,      'USD', 'purchase', TRUE,  95,  '2026-01-18'),
  (1010, 'ACC-010', 'James Miller',  'eu',      150000.00, 'EUR', 'transfer', FALSE, 61,  '2026-01-19')
""")


In [ ]:
spark.sql(f"""
-- Tag region and fraud_flag columns for ABAC policy matching
ALTER TABLE {catalog}.{schema}.transactions ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.transactions ALTER COLUMN fraud_flag SET TAGS ('abac_tut_emergency' = '')
""")


In [ ]:
spark.sql(f"""
-- Fraud analysts: only see fraud-flagged transactions
-- All other analysts: blocked from fraud data
CREATE OR REPLACE FUNCTION {catalog}.{schema}.fraud_row_filter(fraud_flag BOOLEAN)
RETURNS BOOLEAN
RETURN fraud_flag = TRUE OR is_account_group_member('abac_tut_finance_team')
""")


In [ ]:
spark.sql(f"""
-- High-value transaction filter: compliance team sees transactions > $10,000
CREATE OR REPLACE FUNCTION {catalog}.{schema}.high_value_filter(amount DOUBLE)
RETURNS BOOLEAN
RETURN amount > 10000.0 OR is_account_group_member('abac_tut_hr_team')
""")


In [ ]:
spark.sql(f"""
-- Fraud detection policy: abac_tut_finance_team bypass the filter, others only see fraud=TRUE
CREATE OR REPLACE POLICY fraud_detection_filter
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.fraud_row_filter
TO `account users` EXCEPT abac_tut_hr_team, abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_emergency') AS fraud_col
USING COLUMNS (fraud_col)
""")


In [ ]:
spark.sql(f"""
-- Verify fraud filter: abac_tut_finance_team see all rows; others only see fraud_flag=TRUE
SELECT txn_id, account_id, region, amount, fraud_flag, risk_score, txn_date
FROM {catalog}.{schema}.transactions
ORDER BY txn_date
""")


**Expected results by role:**

| Role | Visible Transactions | Reason |
|------|---------------------|--------|
| `abac_tut_finance_team` | All 10 transactions | Exempt from fraud filter |
| `abac_tut_hr_team` | Txns 1002, 1004, 1006, 1008, 1010 (>$10K) | Exempt from fraud filter, see high-value |
| `abac_tut_admins` | All 10 transactions | Exempt from all filters |
| Standard user | Txns 1003, 1004, 1007, 1009 (fraud=TRUE) | Only fraud-flagged visible |